In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.gold.dim_date")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.dim_date (
  date_key BIGINT,
  date TIMESTAMP,
  year INT,
  month INT,
  day INT
)
""")



In [0]:
import pandas as pd

df = spark.table("workspace.silver.movies").toPandas()

dates = pd.to_datetime(df["release_date"], errors="coerce")

dim_date = (
    pd.DataFrame({"date": dates})
    .dropna()
    .drop_duplicates()
)

dim_date["date_key"] = dim_date["date"].dt.strftime("%Y%m%d").astype("int")
dim_date["year"] = dim_date["date"].dt.year
dim_date["month"] = dim_date["date"].dt.month
dim_date["day"] = dim_date["date"].dt.day

df_spark = spark.createDataFrame(dim_date)

In [0]:
# Usamos overwrite para reemplazar completamente los datos existentes
df_spark.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_date")